In [ ]:
# Colab setup - run this first, then switch the runtime to R
# (Runtime > Change runtime type > R). Precompiled Linux binaries via Posit P3M
# keep the install to a couple of minutes instead of compiling from source.
local({
  cn <- tryCatch(system("lsb_release -cs", intern = TRUE), error = function(e) "jammy")
  options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", cn)),
          HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
            paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
})
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(
  c("SingleCellExperiment", "SummarizedExperiment", "MultiAssayExperiment",
    "S4Vectors", "basilisk"),
  update = FALSE, ask = FALSE)
install.packages(c("remotes", "reticulate", "Matrix", "ggplot2"))
remotes::install_github("PYangLab/M3", subdir = "m3-r")
# The first m3_train() builds the bundled Python engine via basilisk (a few
# minutes on first run); afterwards it is cached for the session.

# Data augmentation

m3’s decoder is generative: by sampling the posterior of real cells we synthesise new **cells** and new **donors per condition**, useful for augmenting small cohorts. We train the integration model, then use `m3_augment()` (new donors per condition) and `m3_generate()` (posterior-resampled cells), and check the synthetic data matches the real distribution.

## 1. Load the demo dataset

In [ ]:
library(m3)
library(ggplot2)
set.seed(0)
data <- m3_demo()
data

## 2. Train the integration model

Augmentation only needs the generator, so we skip the donor predictor for speed.

In [ ]:
#To save time, users can set max_epochs to 100 for test.
model <- m3_train(
  data,
  condition_keys   = c("cond_group", "Age_interval"),
  target_condition = "cond_group",
  celltype_key     = "mergedcelltype",
  batch_key        = "batch",
  donor_key        = "sample_id",
  embedding_dim    = 30L,
  donor_prediction = FALSE,
  max_epochs       = 500L,
  seed = 0L
)

## 3. Synthesize new donors per condition

`m3_augment()` resamples real donor templates per condition through the VAE posterior, producing fresh synthetic donors with per-cell-type expression.

In [ ]:
aug <- m3_augment(model, conditions = c("HC", "Severe"), n_donors = c(3L, 3L), tau = 0.8)
syn_rna <- aug$expression$rna
cat("synthetic cells:", paste(dim(syn_rna), collapse = "x"), "\n")

print(table(aug$obs$cond_group))

## 4. Real vs synthetic — sample-level, batch-stratified UMAP

Per batch and per modality (RNA, ADT) we plot one dot per **sample**. Real samples are open circles; m3-generated samples are crosses; colour marks condition. Synthetic samples should cluster with real samples of the same condition within the same batch.

In [ ]:
real_recon <- m3_reconstruct(model, remove_batch = FALSE)$rna
real_recon[is.na(real_recon)] <- 0
m_real <- colMeans(real_recon)
m_syn  <- colMeans(replace(syn_rna, is.na(syn_rna), 0))
pearson <- stats::cor(m_real, m_syn)
cat(sprintf("per-gene mean expression Pearson r = %.3f\n", pearson))

In [ ]:
# Plot real samples in the model's reconstruction (decoder) space — the SAME space
# the generated samples live in — so real and generated of the same condition overlap.
# (Raw counts vs decoder output would separate real/generated by space, not condition.)
recon <- m3_reconstruct(model, remove_batch = FALSE)
obs <- m3_cell_metadata(model)          # row-aligned with the reconstruction
real_rna_all <- replace(recon$rna, is.na(recon$rna), 0)
real_adt_all <- replace(recon$adt, is.na(recon$adt), 0)

per_sample_mean <- function(X, keys) {
  uk <- unique(keys)
  list(M = do.call(rbind, lapply(uk, function(k) colMeans(X[keys == k, , drop = FALSE]))),
       keys = uk)
}
l2 <- function(X) { n <- sqrt(rowSums(X^2)); X / ifelse(n == 0, 1, n) }
embed2 <- function(X, dev) {
  X <- l2(X)
  if (nrow(X) > 4) m3_umap(X, method = "umap", n_neighbors = 15L, min_dist = 0.5,
                           spread = 1.5, metric = "cosine", random_state = 0L, device = dev)
  else m3_pca2(X, device = dev)
}

batches <- sort(unique(as.character(obs$batch)))
N_DONORS <- 5L
big <- list()
for (ci in seq_along(batches)) {
  b <- batches[ci]; inb <- as.character(obs$batch) == b
  rk <- as.character(obs$sample_id[inb])
  rs_rna <- per_sample_mean(real_rna_all[inb, , drop = FALSE], rk)
  rs_adt <- per_sample_mean(real_adt_all[inb, , drop = FALSE], rk)
  scond <- vapply(rs_rna$keys, function(s)
    as.character(obs$cond_group[inb][match(s, rk)]), character(1))

  ag <- m3_augment(model, conditions = c("HC", "Severe"),
                   n_donors = c(N_DONORS, N_DONORS), batch = b, tau = 0.8, seed = 42L + ci - 1L)
  sk <- paste0(ag$obs$cond_group, "/", ag$obs$donor)
  ss_rna <- per_sample_mean(ag$expression$rna, sk)
  ss_adt <- per_sample_mean(ag$expression$adt, sk)
  scond_syn <- sub("/.*$", "", ss_rna$keys)

  for (md in c("RNA", "ADT")) {
    rm_real <- if (md == "RNA") rs_rna$M else rs_adt$M
    rm_syn  <- if (md == "RNA") ss_rna$M else ss_adt$M
    xy <- embed2(rbind(rm_real, rm_syn), model$device)
    nr <- nrow(rm_real)
    big[[length(big) + 1L]] <- rbind(
      data.frame(UMAP1 = xy[seq_len(nr), 1], UMAP2 = xy[seq_len(nr), 2],
                 cond = scond, kind = "real",
                 panel = paste0("Batch ", b, " (", md, ")")),
      data.frame(UMAP1 = xy[-seq_len(nr), 1], UMAP2 = xy[-seq_len(nr), 2],
                 cond = scond_syn, kind = "generated",
                 panel = paste0("Batch ", b, " (", md, ")")))
  }
}
plot_df <- do.call(rbind, big)
cols <- c(HC = "#E64B35", Severe = "#4DBBD5")
ggplot() +
  geom_point(data = subset(plot_df, kind == "real"),
             aes(UMAP1, UMAP2, colour = cond), shape = 1, size = 3, stroke = 1.2) +
  geom_point(data = subset(plot_df, kind == "generated"),
             aes(UMAP1, UMAP2, colour = cond), shape = 4, size = 3, stroke = 1.4) +
  scale_colour_manual(values = cols, name = NULL) +
  facet_wrap(~ panel, scales = "free", ncol = length(batches)) +
  labs(title = "Liu et al — sample-level UMAP: real (o) vs m3 generated (x)") +
  theme_classic() +
  theme(axis.text = element_blank(), axis.ticks = element_blank())

In [ ]:
ggplot(data.frame(real = m_real, syn = m_syn), aes(real, syn)) +
  geom_point(size = 0.6, alpha = 0.4) +
  geom_abline(linetype = "dashed", colour = "grey") +
  labs(x = "real (reconstruction) mean", y = "synthetic mean",
       title = sprintf("Per-gene mean expression (r=%.3f)", pearson)) +
  theme_classic()

## 5. Posterior-resampled cells (`generate`)

`m3_generate()` returns one synthetic cell per reference cell (1:1 posterior resample), handy for noise-augmenting a training set.

In [ ]:
gen <- m3_generate(model, tau = 0.8)
cat("generated:", paste(names(gen), sapply(gen, function(m) paste(dim(m), collapse = "x")),
                        sep = "=", collapse = "  "), "\n")

**Done.** Synthetic donors per condition + posterior-resampled cells, with a real-vs-synthetic UMAP confirming the synthetic cells track the real manifold.